In [2]:

"""
DUAL LAYER IDS - ARP SPOOFING LOAO TEST
Only Label 1 (ARP Spoofing) samples fed through full pipeline

Author: Tan Yi Feng
Student ID: 23WMR14766
"""

import pandas as pd
import numpy as np
import joblib
import yaml
import sys
import pickle
import logging
import time
import threading
import psutil
import os
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from pathlib import Path
import joblib
import pickle
import pandas as pd


SCRIPT_DIR = Path.cwd()


DATASET_PATH = SCRIPT_DIR / 'data' / 'processed' / 'full_inference_cleaned_casing_preserved.csv'
CONFIG_PATH = SCRIPT_DIR / 'config' / 'config.yaml'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'
SRC_PATH = SCRIPT_DIR / 'src'

sys.path.append(str(SRC_PATH))

from preprocessing.feature_extractor import FeatureExtractor

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
logger = logging.getLogger(__name__)

ATTACK_LABELS = {
    1: "ARP Spoofing",
    2: "MQTT Connect Flood", 3: "MQTT Publish Flood",
    4: "MQTT Malformed",     5: "Reconnaissance",
    6: "Recon (VulnScan)",   7: "ICMP Flood",
    8: "SYN Flood",          9: "TCP Flood",
    10: "UDP Flood"
}


class ResourceMonitor:
    def __init__(self, interval=0.1):
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self._running = False
        self._thread = None
        self.peak_memory_mb = 0.0
        self.peak_cpu_percent = 0.0
        self._cpu_samples = []

    def _monitor(self):
        self.process.cpu_percent(interval=None)
        while self._running:
            try:
                mem_mb = self.process.memory_info().rss / (1024 ** 2)
                cpu = self.process.cpu_percent(interval=None)
                if mem_mb > self.peak_memory_mb:
                    self.peak_memory_mb = mem_mb
                if cpu > self.peak_cpu_percent:
                    self.peak_cpu_percent = cpu
                self._cpu_samples.append(cpu)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                break
            time.sleep(self.interval)

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._monitor, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)

    @property
    def avg_cpu_percent(self):
        return np.mean(self._cpu_samples) if self._cpu_samples else 0.0


def load_config(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)



def run_arp_loao_test(dataset_path):
    print("🚀 STARTING ARP SPOOFING LOAO TEST...")
    print("   Layer 2 was trained WITHOUT ARP Spoofing (Label 1)")
    print("   Goal: See how the pipeline handles a completely unseen attack class")

    total_start_time = time.time()
    monitor = ResourceMonitor(interval=0.05)
    monitor.start()

   
    try:
        config = load_config(CONFIG_PATH)
    except Exception as e:
        print(f"❌ Config Load Error: {e}")
        monitor.stop()
        return

  
    print("\n[STEP 1] Loading Models...")
    try:
        ensemble  = joblib.load(MODELS_DIR / 'ensemble.joblib')
        scaler    = joblib.load(MODELS_DIR / 'scaler.joblib')
        
        rf_pipeline       = joblib.load(MODELS_DIR / 'best_rf_pipeline_noARP.pkl')
        with open(MODELS_DIR / 'selected_features_noARP.pkl', 'rb') as f:
            selected_features = list(pickle.load(f))
        print(f"✅ Models loaded (noARP RF pipeline).")
        print(f"   - L2 Features : {len(selected_features)}")
    except FileNotFoundError as e:
        print(f"❌ Model not found: {e}")
        monitor.stop()
        return

   
    print(f"\n[STEP 2] Loading ARP Spoofing samples only (Label 1)...")
    try:
        df_full = pd.read_csv(dataset_path)
        df = df_full[df_full['label'] == 1].copy().reset_index(drop=True)
        total_packets = len(df)
        print(f"   - Total ARP Spoofing samples : {total_packets:,}")
    except Exception as e:
        print(f"❌ Data Load Error: {e}")
        monitor.stop()
        return

    print("\n[STEP 3] Running Layer 1 (Binary Anomaly Detection)...")
    layer1_start = time.time()

    extractor = FeatureExtractor()
    features_df   = extractor.extract_from_dataframe(df)
    features_norm, _ = extractor.normalize_features(features_df, scaler)
    X_L1 = extractor.get_feature_vector(features_norm)

    predictions   = ensemble.predict_with_details(X_L1)
    threshold     = config.get('ensemble', {}).get('threshold_low', 0.1)
    layer1_flags  = predictions['ensemble_score'] > threshold

    suspicious_indices  = np.where(layer1_flags)[0]
    normal_indices      = np.where(~layer1_flags)[0]

    layer1_time = time.time() - layer1_start

   
    flagged_as_attack   = int(layer1_flags.sum())
    missed_by_layer1    = int((~layer1_flags).sum())

    print(f"\n{'='*50}")
    print(f"   LAYER 1 RESULTS — ARP SPOOFING ONLY")
    print(f"{'='*50}")
    print(f"Total ARP Samples          : {total_packets:,}")
    print(f"Flagged as Attack (→ L2)   : {flagged_as_attack:,}  ({flagged_as_attack/total_packets*100:.2f}%)")
    print(f"Missed (classified Normal) : {missed_by_layer1:,}  ({missed_by_layer1/total_packets*100:.2f}%)")
    print(f"Layer 1 Detection Rate     : {flagged_as_attack/total_packets*100:.2f}%")
    print(f"Layer 1 Time               : {layer1_time:.4f} s")

   
    print(f"\n[STEP 4] Running Layer 2 (noARP RF + Confidence Threshold)...")
    layer2_start = time.time()

    l2_decisions = np.full(total_packets, -1, dtype=int)  # -1 = never reached L2

    if len(suspicious_indices) > 0:
        try:
            X_L2_subset = df.loc[suspicious_indices, selected_features].copy()
            X_L2_subset.replace([np.inf, -np.inf], np.nan, inplace=True)
            imputer  = SimpleImputer(strategy='median')
            X_L2_clean = imputer.fit_transform(X_L2_subset)

            probs     = rf_pipeline.predict_proba(X_L2_clean)
            max_probs = np.max(probs, axis=1)
            raw_preds = np.argmax(probs, axis=1)

           
            mapped_preds = raw_preds + 2

            
            thresholded = np.where(max_probs >= 0.9, mapped_preds, 99)
            l2_decisions[suspicious_indices] = thresholded

        except KeyError as e:
            print(f"❌ Column Error: {e}")
            monitor.stop()
            return

    layer2_time = time.time() - layer2_start

   
    reached_l2      = l2_decisions[suspicious_indices]
    ambiguous_mask  = reached_l2 == 99
    confident_mask  = reached_l2 != 99

    ambiguous_count = int(ambiguous_mask.sum())
    confident_count = int(confident_mask.sum())

    print(f"\n{'='*50}")
    print(f"   LAYER 2 RESULTS — ARP SAMPLES THAT REACHED L2")
    print(f"{'='*50}")
    print(f"Samples reaching Layer 2   : {len(suspicious_indices):,}")
    print(f"Confidence Threshold       : 0.9")
    print(f"")
    print(f"Flagged Ambiguous (Lb 99)  : {ambiguous_count:,}  ({ambiguous_count/len(suspicious_indices)*100:.2f}%)")
    print(f"Confidently Mislabelled    : {confident_count:,}  ({confident_count/len(suspicious_indices)*100:.2f}%)")

   
    arp_max_probs = np.max(rf_pipeline.predict_proba(
        SimpleImputer(strategy='median').fit_transform(
            df.loc[suspicious_indices, selected_features].replace([np.inf, -np.inf], np.nan)
        )
    ), axis=1)

    print(f"\nConfidence Score Stats (ARP at L2):")
    print(f"  Mean   : {arp_max_probs.mean():.4f}")
    print(f"  Median : {np.median(arp_max_probs):.4f}")
    print(f"  Min    : {arp_max_probs.min():.4f}")
    print(f"  Max    : {arp_max_probs.max():.4f}")

    
    if confident_count > 0:
        confident_preds = reached_l2[confident_mask]
        unique_preds, counts = np.unique(confident_preds, return_counts=True)
        print(f"\nARP Mislabelled As (Confident):")
        for pred_label, cnt in zip(unique_preds, counts):
            name = ATTACK_LABELS.get(int(pred_label), f"Class {pred_label}")
            print(f"  → {name:<25} : {cnt:,}  ({cnt/confident_count*100:.2f}%)")

  
    monitor.stop()
    total_time = time.time() - total_start_time

   
    missed_l1        = missed_by_layer1
    ambiguous_total  = ambiguous_count
    mislabelled      = confident_count

    print(f"\n{'='*50}")
    print(f"   FULL PIPELINE — ARP FATE SUMMARY")
    print(f"{'='*50}")
    print(f"Total ARP Samples          : {total_packets:,}  (100%)")
    print(f"")
    print(f"Missed by Layer 1          : {missed_l1:,}  ({missed_l1/total_packets*100:.2f}%)  ← classified as Normal")
    print(f"Flagged Ambiguous (L2)     : {ambiguous_total:,}  ({ambiguous_total/total_packets*100:.2f}%)  ← novel threat signal")
    print(f"Confidently Mislabelled    : {mislabelled:,}  ({mislabelled/total_packets*100:.2f}%)  ← wrong known label")
    print(f"")
    correctly_handled = flagged_as_attack  # L1 caught them at minimum
    print(f"Detected as threat (L1+L2) : {correctly_handled:,}  ({correctly_handled/total_packets*100:.2f}%)")
    print(f"  of which → Ambiguous     : {ambiguous_total:,}  ({ambiguous_total/total_packets*100:.2f}%)")
    print(f"  of which → Mislabelled   : {mislabelled:,}  ({mislabelled/total_packets*100:.2f}%)")

   
    avg_latency = total_time / total_packets
    print(f"\n{'='*50}")
    print(f"           PERFORMANCE METRICS")
    print(f"{'='*50}")
    print(f"Total Packets         : {total_packets:,}")
    print(f"Layer 1 Time          : {layer1_time:.4f} s")
    print(f"Layer 2 Time          : {layer2_time:.4f} s")
    print(f"Total Time            : {total_time:.4f} s")
    print(f"Avg Latency/Packet    : {avg_latency:.8f} s  ({avg_latency*1000:.4f} ms)")
    print(f"Throughput            : {total_packets/total_time:.2f} packets/s")

    print(f"\n{'='*50}")
    print(f"         RESOURCE UTILIZATION")
    print(f"{'='*50}")
    print(f"Peak Memory Usage     : {monitor.peak_memory_mb:.2f} MB")
    print(f"Peak CPU Utilization  : {monitor.peak_cpu_percent:.2f}%")
    print(f"Avg  CPU Utilization  : {monitor.avg_cpu_percent:.2f}%")

    print("\n✅ ARP LOAO Test Finished.")

run_arp_loao_test(DATASET_PATH)

🚀 STARTING ARP SPOOFING LOAO TEST...
   Layer 2 was trained WITHOUT ARP Spoofing (Label 1)
   Goal: See how the pipeline handles a completely unseen attack class

[STEP 1] Loading Models...
✅ Models loaded (noARP RF pipeline).
   - L2 Features : 23

[STEP 2] Loading ARP Spoofing samples only (Label 1)...


Extracting features from 16,046 flows...
Input columns: 41


   - Total ARP Spoofing samples : 16,046

[STEP 3] Running Layer 1 (Binary Anomaly Detection)...


[OK] Extracted 67 columns
     - 63 ML features
     - Metadata: timestamp, protocol, label, scenario
     - NO synthetic IPs/ports (removed)
[OK] Features normalized (detection mode - used existing scaler)



   LAYER 1 RESULTS — ARP SPOOFING ONLY
Total ARP Samples          : 16,046
Flagged as Attack (→ L2)   : 16,024  (99.86%)
Missed (classified Normal) : 22  (0.14%)
Layer 1 Detection Rate     : 99.86%
Layer 1 Time               : 3.3629 s

[STEP 4] Running Layer 2 (noARP RF + Confidence Threshold)...


C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



   LAYER 2 RESULTS — ARP SAMPLES THAT REACHED L2
Samples reaching Layer 2   : 16,024
Confidence Threshold       : 0.9

Flagged Ambiguous (Lb 99)  : 5,139  (32.07%)
Confidently Mislabelled    : 10,885  (67.93%)


C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



Confidence Score Stats (ARP at L2):
  Mean   : 0.8859
  Median : 0.9938
  Min    : 0.3395
  Max    : 1.0000

ARP Mislabelled As (Confident):
  → MQTT Malformed            : 3,641  (33.45%)
  → Reconnaissance            : 6,702  (61.57%)
  → Recon (VulnScan)          : 542  (4.98%)

   FULL PIPELINE — ARP FATE SUMMARY
Total ARP Samples          : 16,046  (100%)

Missed by Layer 1          : 22  (0.14%)  ← classified as Normal
Flagged Ambiguous (L2)     : 5,139  (32.03%)  ← novel threat signal
Confidently Mislabelled    : 10,885  (67.84%)  ← wrong known label

Detected as threat (L1+L2) : 16,024  (99.86%)
  of which → Ambiguous     : 5,139  (32.03%)
  of which → Mislabelled   : 10,885  (67.84%)

           PERFORMANCE METRICS
Total Packets         : 16,046
Layer 1 Time          : 3.3629 s
Layer 2 Time          : 0.4521 s
Total Time            : 42.6894 s
Avg Latency/Packet    : 0.00266044 s  (2.6604 ms)
Throughput            : 375.88 packets/s

         RESOURCE UTILIZATION
Peak Memory 

In [1]:

"""
DUAL LAYER IDS - UDP FLood LOAO TEST
Only Label 10 (UDP Flood) samples fed through full pipeline

Author: Tan Yi Feng
Student ID: 23WMR14766
"""


import pandas as pd
import numpy as np
import joblib
import yaml
import sys
import pickle
import logging
import time
import threading
import psutil
import os
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from pathlib import Path
import joblib
import pickle
import pandas as pd

SCRIPT_DIR = Path.cwd()

DATASET_PATH = SCRIPT_DIR / 'data' / 'processed' / 'full_inference_cleaned_casing_preserved.csv'
CONFIG_PATH = SCRIPT_DIR / 'config' / 'config.yaml'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'
SRC_PATH = SCRIPT_DIR / 'src'

sys.path.append(str(SRC_PATH))

from preprocessing.feature_extractor import FeatureExtractor

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)
logger = logging.getLogger(__name__)

ATTACK_LABELS = {
    1: "ARP Spoofing",
    2: "MQTT Connect Flood", 3: "MQTT Publish Flood",
    4: "MQTT Malformed",     5: "Reconnaissance",
    6: "Recon (VulnScan)",   7: "ICMP Flood",
    8: "SYN Flood",          9: "TCP Flood",
    10: "UDP Flood"
}

class ResourceMonitor:
    def __init__(self, interval=0.1):
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self._running = False
        self._thread = None
        self.peak_memory_mb = 0.0
        self.peak_cpu_percent = 0.0
        self._cpu_samples = []

    def _monitor(self):
        self.process.cpu_percent(interval=None)
        while self._running:
            try:
                mem_mb = self.process.memory_info().rss / (1024 ** 2)
                cpu = self.process.cpu_percent(interval=None)
                if mem_mb > self.peak_memory_mb:
                    self.peak_memory_mb = mem_mb
                if cpu > self.peak_cpu_percent:
                    self.peak_cpu_percent = cpu
                self._cpu_samples.append(cpu)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                break
            time.sleep(self.interval)

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._monitor, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=2)

    @property
    def avg_cpu_percent(self):
        return np.mean(self._cpu_samples) if self._cpu_samples else 0.0


def load_config(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


def run_arp_loao_test(dataset_path):
    print("🚀 STARTING UDP Flood LOAO TEST...")
    print("   Layer 2 was trained WITHOUT UDP (Label 10)")
    print("   Goal: See how the pipeline handles a completely unseen attack class")

    total_start_time = time.time()
    monitor = ResourceMonitor(interval=0.05)
    monitor.start()

    try:
        config = load_config(CONFIG_PATH)
    except Exception as e:
        print(f"❌ Config Load Error: {e}")
        monitor.stop()
        return

    print("\n[STEP 1] Loading Models...")
    try:
        ensemble  = joblib.load(MODELS_DIR / 'ensemble.joblib')
        scaler    = joblib.load(MODELS_DIR / 'scaler.joblib')
        rf_pipeline       = joblib.load(MODELS_DIR / 'best_rf_pipeline_noUDP.pkl')
        with open(MODELS_DIR / 'selected_features_noUDP.pkl', 'rb') as f:
            selected_features = list(pickle.load(f))
        print(f"✅ Models loaded (noARP RF pipeline).")
        print(f"   - L2 Features : {len(selected_features)}")
    except FileNotFoundError as e:
        print(f"❌ Model not found: {e}")
        monitor.stop()
        return

    print(f"\n[STEP 2] Loading ARP Spoofing samples only (Label 1)...")
    try:
        df_full = pd.read_csv(dataset_path)
        df = df_full[df_full['label'] == 10].copy().reset_index(drop=True)
        total_packets = len(df)
        print(f"   - Total ARP Spoofing samples : {total_packets:,}")
    except Exception as e:
        print(f"❌ Data Load Error: {e}")
        monitor.stop()
        return

    print("\n[STEP 3] Running Layer 1 (Binary Anomaly Detection)...")
    layer1_start = time.time()

    extractor = FeatureExtractor()
    features_df   = extractor.extract_from_dataframe(df)
    features_norm, _ = extractor.normalize_features(features_df, scaler)
    X_L1 = extractor.get_feature_vector(features_norm)

    predictions   = ensemble.predict_with_details(X_L1)
    threshold     = config.get('ensemble', {}).get('threshold_low', 0.1)
    layer1_flags  = predictions['ensemble_score'] > threshold

    suspicious_indices  = np.where(layer1_flags)[0]
    normal_indices      = np.where(~layer1_flags)[0]

    layer1_time = time.time() - layer1_start

    flagged_as_attack   = int(layer1_flags.sum())
    missed_by_layer1    = int((~layer1_flags).sum())

    print(f"\n{'='*50}")
    print(f"   LAYER 1 RESULTS — UDP Flood ONLY")
    print(f"{'='*50}")
    print(f"Total UDP Samples          : {total_packets:,}")
    print(f"Flagged as Attack (→ L2)   : {flagged_as_attack:,}  ({flagged_as_attack/total_packets*100:.2f}%)")
    print(f"Missed (classified Normal) : {missed_by_layer1:,}  ({missed_by_layer1/total_packets*100:.2f}%)")
    print(f"Layer 1 Detection Rate     : {flagged_as_attack/total_packets*100:.2f}%")
    print(f"Layer 1 Time               : {layer1_time:.4f} s")

    print(f"\n[STEP 4] Running Layer 2 (noUDP RF + Confidence Threshold)...")
    layer2_start = time.time()

    l2_decisions = np.full(total_packets, -1, dtype=int) 

    if len(suspicious_indices) > 0:
        try:
            X_L2_subset = df.loc[suspicious_indices, selected_features].copy()
            X_L2_subset.replace([np.inf, -np.inf], np.nan, inplace=True)
            imputer  = SimpleImputer(strategy='median')
            X_L2_clean = imputer.fit_transform(X_L2_subset)

            probs     = rf_pipeline.predict_proba(X_L2_clean)
            max_probs = np.max(probs, axis=1)
            raw_preds = np.argmax(probs, axis=1)

            mapped_preds = raw_preds + 1

          
            thresholded = np.where(max_probs >= 0.9, mapped_preds, 99)
            l2_decisions[suspicious_indices] = thresholded

        except KeyError as e:
            print(f"❌ Column Error: {e}")
            monitor.stop()
            return

    layer2_time = time.time() - layer2_start

    reached_l2      = l2_decisions[suspicious_indices]
    ambiguous_mask  = reached_l2 == 99
    confident_mask  = reached_l2 != 99

    ambiguous_count = int(ambiguous_mask.sum())
    confident_count = int(confident_mask.sum())

    print(f"\n{'='*50}")
    print(f"   LAYER 2 RESULTS — UDP SAMPLES THAT REACHED L2")
    print(f"{'='*50}")
    print(f"Samples reaching Layer 2   : {len(suspicious_indices):,}")
    print(f"Confidence Threshold       : 0.9")
    print(f"")
    print(f"Flagged Ambiguous (Lb 99)  : {ambiguous_count:,}  ({ambiguous_count/len(suspicious_indices)*100:.2f}%)")
    print(f"Confidently Mislabelled    : {confident_count:,}  ({confident_count/len(suspicious_indices)*100:.2f}%)")

    arp_max_probs = np.max(rf_pipeline.predict_proba(
        SimpleImputer(strategy='median').fit_transform(
            df.loc[suspicious_indices, selected_features].replace([np.inf, -np.inf], np.nan)
        )
    ), axis=1)

    print(f"\nConfidence Score Stats (ARP at L2):")
    print(f"  Mean   : {arp_max_probs.mean():.4f}")
    print(f"  Median : {np.median(arp_max_probs):.4f}")
    print(f"  Min    : {arp_max_probs.min():.4f}")
    print(f"  Max    : {arp_max_probs.max():.4f}")

    if confident_count > 0:
        confident_preds = reached_l2[confident_mask]
        unique_preds, counts = np.unique(confident_preds, return_counts=True)
        print(f"\nUDP Mislabelled As (Confident):")
        for pred_label, cnt in zip(unique_preds, counts):
            name = ATTACK_LABELS.get(int(pred_label), f"Class {pred_label}")
            print(f"  → {name:<25} : {cnt:,}  ({cnt/confident_count*100:.2f}%)")

    # ==========================================
    # OVERALL PIPELINE SUMMARY FOR ARP
    # ==========================================
    monitor.stop()
    total_time = time.time() - total_start_time

    # Categorise every ARP sample's final fate
    missed_l1        = missed_by_layer1
    ambiguous_total  = ambiguous_count
    mislabelled      = confident_count

    print(f"\n{'='*50}")
    print(f"   FULL PIPELINE — UDP FATE SUMMARY")
    print(f"{'='*50}")
    print(f"Total UDP Samples          : {total_packets:,}  (100%)")
    print(f"")
    print(f"Missed by Layer 1          : {missed_l1:,}  ({missed_l1/total_packets*100:.2f}%)  ← classified as Normal")
    print(f"Flagged Ambiguous (L2)     : {ambiguous_total:,}  ({ambiguous_total/total_packets*100:.2f}%)  ← novel threat signal")
    print(f"Confidently Mislabelled    : {mislabelled:,}  ({mislabelled/total_packets*100:.2f}%)  ← wrong known label")
    print(f"")
    correctly_handled = flagged_as_attack  # L1 caught them at minimum
    print(f"Detected as threat (L1+L2) : {correctly_handled:,}  ({correctly_handled/total_packets*100:.2f}%)")
    print(f"  of which → Ambiguous     : {ambiguous_total:,}  ({ambiguous_total/total_packets*100:.2f}%)")
    print(f"  of which → Mislabelled   : {mislabelled:,}  ({mislabelled/total_packets*100:.2f}%)")

    # ==========================================
    # PERFORMANCE
    # ==========================================
    avg_latency = total_time / total_packets
    print(f"\n{'='*50}")
    print(f"           PERFORMANCE METRICS")
    print(f"{'='*50}")
    print(f"Total Packets         : {total_packets:,}")
    print(f"Layer 1 Time          : {layer1_time:.4f} s")
    print(f"Layer 2 Time          : {layer2_time:.4f} s")
    print(f"Total Time            : {total_time:.4f} s")
    print(f"Avg Latency/Packet    : {avg_latency:.8f} s  ({avg_latency*1000:.4f} ms)")
    print(f"Throughput            : {total_packets/total_time:.2f} packets/s")

    print(f"\n{'='*50}")
    print(f"         RESOURCE UTILIZATION")
    print(f"{'='*50}")
    print(f"Peak Memory Usage     : {monitor.peak_memory_mb:.2f} MB")
    print(f"Peak CPU Utilization  : {monitor.peak_cpu_percent:.2f}%")
    print(f"Avg  CPU Utilization  : {monitor.avg_cpu_percent:.2f}%")

    print("\n✅ ARP LOAO Test Finished.")

run_arp_loao_test(DATASET_PATH)

🚀 STARTING ARP SPOOFING LOAO TEST...
   Layer 2 was trained WITHOUT UDP (Label 10)
   Goal: See how the pipeline handles a completely unseen attack class

[STEP 1] Loading Models...
✅ Models loaded (noARP RF pipeline).
   - L2 Features : 28

[STEP 2] Loading ARP Spoofing samples only (Label 1)...


Extracting features from 2,202,888 flows...
Input columns: 41


   - Total ARP Spoofing samples : 2,202,888

[STEP 3] Running Layer 1 (Binary Anomaly Detection)...


[OK] Extracted 67 columns
     - 63 ML features
     - Metadata: timestamp, protocol, label, scenario
     - NO synthetic IPs/ports (removed)
C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
[OK] Features normalized (detection mode - used existing scaler)



   LAYER 1 RESULTS — ARP SPOOFING ONLY
Total ARP Samples          : 2,202,888
Flagged as Attack (→ L2)   : 2,202,364  (99.98%)
Missed (classified Normal) : 524  (0.02%)
Layer 1 Detection Rate     : 99.98%
Layer 1 Time               : 1457.5860 s

[STEP 4] Running Layer 2 (noARP RF + Confidence Threshold)...


C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



   LAYER 2 RESULTS — UDP SAMPLES THAT REACHED L2
Samples reaching Layer 2   : 2,202,364
Confidence Threshold       : 0.9

Flagged Ambiguous (Lb 99)  : 2,202,362  (100.00%)
Confidently Mislabelled    : 2  (0.00%)


C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(



Confidence Score Stats (ARP at L2):
  Mean   : 0.8510
  Median : 0.8647
  Min    : 0.1579
  Max    : 0.9699

UDP Mislabelled As (Confident):
  → MQTT Connect Flood        : 1  (50.00%)
  → MQTT Malformed            : 1  (50.00%)

   FULL PIPELINE — UDP FATE SUMMARY
Total UDP Samples          : 2,202,888  (100%)

Missed by Layer 1          : 524  (0.02%)  ← classified as Normal
Flagged Ambiguous (L2)     : 2,202,362  (99.98%)  ← novel threat signal
Confidently Mislabelled    : 2  (0.00%)  ← wrong known label

Detected as threat (L1+L2) : 2,202,364  (99.98%)
  of which → Ambiguous     : 2,202,362  (99.98%)
  of which → Mislabelled   : 2  (0.00%)

           PERFORMANCE METRICS
Total Packets         : 2,202,888
Layer 1 Time          : 1457.5860 s
Layer 2 Time          : 43.0141 s
Total Time            : 1589.0888 s
Avg Latency/Packet    : 0.00072137 s  (0.7214 ms)
Throughput            : 1386.26 packets/s

         RESOURCE UTILIZATION
Peak Memory Usage     : 7050.16 MB
Peak CPU Utilizat